In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataCleaning").getOrCreate()

df = spark.read.csv(
    "customers_data.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

+-----------+-------+----+-------+-------------+-------+--------------------+----------+----------+
|customer_id|   name| age| salary|      country|   city|               email|     phone| join_date|
+-----------+-------+----+-------+-------------+-------+--------------------+----------+----------+
|          1|  John | 150|-1000.0|        U.S.A|  Delhi|floresrandall@liv...|9876543210|12-05-2024|
|          1|  John |  25| 5000.0|        U.S.A|  Delhi|                NULL|9876543210|2024/01/01|
|          2|   mike|NULL|   NULL|United States|Chennai| sarah97@hotmail.com|      NULL|31-13-2024|
|          4|   mike|  30|-1000.0|          USA|Chennai|       invalid_email|9876543210|31-13-2024|
|          5|  david|  45| 5000.0|        india|   NULL|                NULL|9876543210|      NULL|
+-----------+-------+----+-------+-------------+-------+--------------------+----------+----------+
only showing top 5 rows
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable

In [2]:
from pyspark.sql.functions import col, explode
from pyspark.sql.types import StructType, ArrayType

def flatten_df(df):
    
    def _flatten(schema, prefix=""):
        fields = []
        for field in schema.fields:
            name = f"{prefix}{field.name}"
            dtype = field.dataType
            if isinstance(dtype, StructType):
                # recurse into struct
                fields.extend(_flatten(dtype, prefix=name + "."))
            elif isinstance(dtype, ArrayType) and isinstance(dtype.elementType, StructType):
                # explode array of structs
                fields.append((name, "explode"))
            else:
                fields.append((name, "column"))
        return fields

    flat_fields = _flatten(df.schema)

    # Explode arrays first
    for f, t in flat_fields:
        if t == "explode":
            df = df.withColumn(f, explode(col(f)))

    # Build select expressions
    select_exprs = []
    for f, t in flat_fields:
        if t == "column":
            select_exprs.append(col(f).alias(f.replace(".", "_")))
        elif t == "explode":
            # flatten struct fields inside exploded array
            struct_fields = df.select(col(f + ".*")).schema.fields
            for sf in struct_fields:
                select_exprs.append(col(f + "." + sf.name).alias(f.replace(".", "_") + "_" + sf.name))

    return df.select(*select_exprs)


In [3]:
nested_df = spark.read.json("nested_orders.json")

flat_df = flatten_df(nested_df)
flat_df.printSchema()
flat_df.show(truncate=False)


root
 |-- customer_address_city: string (nullable = true)
 |-- customer_address_country: string (nullable = true)
 |-- customer_customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- items_price: long (nullable = true)
 |-- items_product: string (nullable = true)
 |-- items_quantity: long (nullable = true)
 |-- order_id: long (nullable = true)

+---------------------+------------------------+--------------------+--------------+--------------------------+-----------+-------------+--------------+--------+
|customer_address_city|customer_address_country|customer_customer_id|customer_name |event_timestamp           |items_price|items_product|items_quantity|order_id|
+---------------------+------------------------+--------------------+--------------+--------------------------+-----------+-------------+--------------+--------+
|Delhi                |INDIA                   |90                  |Lynn Ortiz    |2026-

In [4]:
#Extract product, price, and quantity from exploded struct.

flat_df = flat_df.withColumn("product", col("items_product")) \
    .withColumn("price", col("items_price")) \
    .withColumn("quantity", col("items_quantity")) \
    .drop("items_product", "items_price", "items_quantity")
flat_df.show(20)

+---------------------+------------------------+--------------------+--------------+--------------------+--------+--------+-----+--------+
|customer_address_city|customer_address_country|customer_customer_id| customer_name|     event_timestamp|order_id| product|price|quantity|
+---------------------+------------------------+--------------------+--------------+--------------------+--------+--------+-----+--------+
|                Delhi|                   INDIA|                  90|    Lynn Ortiz|2026-02-02 23:39:...|       1|  Laptop| 1000|      -1|
|                Delhi|                   india|                  36|     David Lee|2026-04-05 23:39:...|       2| Monitor| NULL|       2|
|                Delhi|                   india|                  36|     David Lee|2026-04-05 23:39:...|       2|  Tablet|  500|       1|
|                Delhi|                   india|                  36|     David Lee|2026-04-05 23:39:...|       2|  Laptop| NULL|       2|
|                Delhi|    

In [5]:
#Remove rows having null products and null prices.

cleaned_df = flat_df.filter(col("product").isNotNull() & col("price").isNotNull())
cleaned_df.show(20)

+---------------------+------------------------+--------------------+------------------+--------------------+--------+--------+-----+--------+
|customer_address_city|customer_address_country|customer_customer_id|     customer_name|     event_timestamp|order_id| product|price|quantity|
+---------------------+------------------------+--------------------+------------------+--------------------+--------+--------+-----+--------+
|                Delhi|                   INDIA|                  90|        Lynn Ortiz|2026-02-02 23:39:...|       1|  Laptop| 1000|      -1|
|                Delhi|                   india|                  36|         David Lee|2026-04-05 23:39:...|       2|  Tablet|  500|       1|
|                Delhi|                   India|                  75|    Crystal Gordon|2026-05-01 23:39:...|       3|  Tablet|  500|      -1|
|                Delhi|                   India|                  75|    Crystal Gordon|2026-05-01 23:39:...|       3|Keyboard| 1000|      -1|

In [6]:
#Remove negative quantities from nested JSON.

cleaned_df = cleaned_df.filter(col("quantity") >= 0)
cleaned_df.show(20)

+---------------------+------------------------+--------------------+------------------+--------------------+--------+--------+-----+--------+
|customer_address_city|customer_address_country|customer_customer_id|     customer_name|     event_timestamp|order_id| product|price|quantity|
+---------------------+------------------------+--------------------+------------------+--------------------+--------+--------+-----+--------+
|                Delhi|                   india|                  36|         David Lee|2026-04-05 23:39:...|       2|  Tablet|  500|       1|
|              Chennai|                   india|                  78|      Warren Lewis|2026-01-29 23:39:...|       4|Keyboard|  500|       1|
|                Delhi|                   india|                  31|     Cameron Moore|2026-02-15 23:39:...|       5|Keyboard|  500|       2|
|                Delhi|                   India|                  81|       Tonya Moran|2026-04-01 23:39:...|       7| Monitor| 1000|       2|

In [7]:
#Standardize nested country values into uppercase.

from pyspark.sql.functions import initcap, upper

cleaned_df = cleaned_df.withColumn("customer_address_country", upper(col("customer_address_country")))
cleaned_df.show(20)

+---------------------+------------------------+--------------------+------------------+--------------------+--------+--------+-----+--------+
|customer_address_city|customer_address_country|customer_customer_id|     customer_name|     event_timestamp|order_id| product|price|quantity|
+---------------------+------------------------+--------------------+------------------+--------------------+--------+--------+-----+--------+
|                Delhi|                   INDIA|                  36|         David Lee|2026-04-05 23:39:...|       2|  Tablet|  500|       1|
|              Chennai|                   INDIA|                  78|      Warren Lewis|2026-01-29 23:39:...|       4|Keyboard|  500|       1|
|                Delhi|                   INDIA|                  31|     Cameron Moore|2026-02-15 23:39:...|       5|Keyboard|  500|       2|
|                Delhi|                   INDIA|                  81|       Tonya Moran|2026-04-01 23:39:...|       7| Monitor| 1000|       2|

In [8]:
#Remove duplicate order IDs.

cleaned_df = cleaned_df.dropDuplicates(subset=["order_id"])
cleaned_df.show(20)

+---------------------+------------------------+--------------------+------------------+--------------------+--------+--------+-----+--------+
|customer_address_city|customer_address_country|customer_customer_id|     customer_name|     event_timestamp|order_id| product|price|quantity|
+---------------------+------------------------+--------------------+------------------+--------------------+--------+--------+-----+--------+
|                Delhi|                   INDIA|                  36|         David Lee|2026-04-05 23:39:...|       2|  Tablet|  500|       1|
|              Chennai|                   INDIA|                  78|      Warren Lewis|2026-01-29 23:39:...|       4|Keyboard|  500|       1|
|                Delhi|                   INDIA|                  31|     Cameron Moore|2026-02-15 23:39:...|       5|Keyboard|  500|       2|
|                Delhi|                   INDIA|                  81|       Tonya Moran|2026-04-01 23:39:...|       7| Monitor| 1000|       2|

In [9]:
import pyspark
print(pyspark.__version__)
spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()

4.1.1


'3.4.2'

In [11]:
cleaned_df.write.csv("cleaned_orders_data.csv", header=True, mode="overwrite")